# V2 — Perplexity: generation + evaluation (Colab)

**Required Drive layout:** `Steganography_benchmarks_V2/scripts/` (the full `scripts/` folder from the repo).

Already have RAW only? Copy runs from local `results/perplexity/runs/` → Drive `Steganography_benchmarks_V2/runs/`, set `SKIP_IF_EVALUATED = True` — the notebook will eval only.

Pipeline (`--eval-after`):
1. generates baseline + stego (or reuses existing RAW)
2. scores perplexity with the same model (one GPU load)

Results: `runs/<run>/evaluation/perplexity_results.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)  # on error 107: re-run this cell

import os
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Steganography_benchmarks_V2')
SCRIPTS_DIR = PROJECT_DIR / 'scripts'
MODEL_CACHE_DIR = PROJECT_DIR / 'models_cache'
RUNS_DIR = PROJECT_DIR / 'runs'

for d in (PROJECT_DIR, SCRIPTS_DIR, MODEL_CACHE_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

runner = SCRIPTS_DIR / 'notebook_runner.py'
if not runner.is_file():
    raise FileNotFoundError(
        'Missing scripts/ on Drive.\n'
        'Upload from the repo: Steganography_benchmarks_V2/scripts/ → '
        f'{SCRIPTS_DIR}\n'
        '(The old layout with .py files in the project root no longer works.)'
    )

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = str(MODEL_CACHE_DIR)
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print('PROJECT_DIR:', PROJECT_DIR)
print('scripts OK:', runner)
print('RUNS_DIR:', RUNS_DIR)

In [ ]:
!pip install -q -r scripts/requirements.txt

In [ ]:
from getpass import getpass
import os

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF_TOKEN (Llama/Gemma): ')

In [ ]:
# --- CONFIG ---
MODELS = ['qwen', 'llama', 'gemma']          # model list
THRESHOLDS = [0.0, 0.01, 0.05, 0.1]        # threshold list

TOP_N = 16
MAX_NEW_TOKENS = 512
SKIP_IF_EVALUATED = True     # skip if evaluation/perplexity_results.json exists
CONTINUE_ON_ERROR = True     # continue after a single-run failure

import gc
import json
import sys
from pathlib import Path

import torch

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
from notebook_runner import run_live

TEST = 'perplexity'


def free_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _th_match(a: float, b: float) -> bool:
    return abs(float(a) - float(b)) < 1e-9


def find_run(runs_dir: Path, model: str, threshold: float) -> Path | None:
    """Najnowszy completed run perplexity dla model+threshold."""
    if not runs_dir.exists():
        return None
    candidates: list[tuple[str, Path]] = []
    for manifest_path in runs_dir.glob('*/manifest.json'):
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            continue
        if manifest.get('test') != TEST:
            continue
        if manifest.get('model_key') != model:
            continue
        if not _th_match(manifest.get('threshold', -1), threshold):
            continue
        if manifest.get('status') != 'completed':
            continue
        candidates.append((manifest.get('updated_at', ''), manifest_path.parent))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[-1][1]


def has_eval(run_dir: Path) -> bool:
    return (run_dir / 'evaluation' / 'perplexity_results.json').is_file()


def build_pipeline_cmd(model: str, threshold: float, run_dir: Path | None = None) -> list[str]:
    cmd = [
        sys.executable,
        'scripts/generate_responses.py',
        '--test', TEST,
        '--threshold', str(threshold),
        '--top-n', str(TOP_N),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--platform', 'colab',
        '--output-root', 'runs',
        '--eval-after',
    ]
    if run_dir is not None:
        cmd += ['--run-dir', str(run_dir)]
    else:
        cmd += ['--model', model]
    return cmd


jobs = [(m, th) for m in MODELS for th in THRESHOLDS]
print(f'Plan: {len(jobs)} perplexity runs (gen + eval, 1× GPU load)')
print(f'  models: {MODELS}')
print(f'  thresholds: {THRESHOLDS}')

ok, skipped, failed, results = [], [], [], []

for index, (model, threshold) in enumerate(jobs, start=1):
    label = f'{model} | th={threshold}'
    print('\n' + '=' * 60)
    print(f'[{index}/{len(jobs)}] {label}')
    print('=' * 60)

    free_gpu()
    run_dir = find_run(RUNS_DIR, model, threshold)

    if SKIP_IF_EVALUATED and run_dir is not None and has_eval(run_dir):
        print(f'Skipping — already evaluated: {run_dir.name}')
        skipped.append(label)
        data = json.loads((run_dir / 'evaluation' / 'perplexity_results.json').read_text())
        results.append({
            'model': model,
            'threshold': threshold,
            'run': run_dir.name,
            'baseline_ppl': data.get('baseline_perplexity'),
            'stego_ppl': data.get('perplexity'),
            'delta': data.get('perplexity_delta'),
        })
        continue

    try:
        if run_dir is None:
            print('New run: generation + eval (--eval-after)...')
        else:
            print(f'Existing RAW ({run_dir.name}): eval-only (--eval-after)...')

        run_live(build_pipeline_cmd(model, threshold, run_dir))
        run_dir = run_dir or find_run(RUNS_DIR, model, threshold)
        if run_dir is None or not has_eval(run_dir):
            raise RuntimeError('Missing perplexity_results.json after pipeline finished')

        data = json.loads((run_dir / 'evaluation' / 'perplexity_results.json').read_text())
        results.append({
            'model': model,
            'threshold': threshold,
            'run': run_dir.name,
            'baseline_ppl': data.get('baseline_perplexity'),
            'stego_ppl': data.get('perplexity'),
            'delta': data.get('perplexity_delta'),
        })
        ok.append(label)
        print(
            f"OK: baseline={data['baseline_perplexity']:.2f}, "
            f"stego={data['perplexity']:.2f}, "
            f"delta={data['perplexity_delta']:+.2f}"
        )
    except Exception as exc:
        print(f'ERROR: {exc}')
        failed.append((label, str(exc)))
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        free_gpu()

print('\n' + '#' * 60)
print('SUMMARY')
print('#' * 60)
print(f'OK: {len(ok)} | Skipped: {len(skipped)} | Failed: {len(failed)}')
if failed:
    print('\nFailed runs:')
    for label, err in failed:
        print(f'  - {label}: {err}')

In [ ]:
# Perplexity results table
import json
from pathlib import Path

rows = []
for manifest_path in sorted(Path('runs').glob('*/manifest.json')):
    m = json.loads(manifest_path.read_text(encoding='utf-8'))
    if m.get('test') != 'perplexity':
        continue
    run_dir = manifest_path.parent
    eval_path = run_dir / 'evaluation' / 'perplexity_results.json'
    row = {
        'model': m.get('model_key'),
        'th': m.get('threshold'),
        'status': m.get('status'),
        'run': run_dir.name,
        'eval': eval_path.is_file(),
    }
    if eval_path.is_file():
        d = json.loads(eval_path.read_text())
        row['baseline_ppl'] = d.get('baseline_perplexity')
        row['stego_ppl'] = d.get('perplexity')
        row['delta'] = d.get('perplexity_delta')
    rows.append(row)

rows.sort(key=lambda r: (r['model'], float(r['th'])))
print(f'Perplexity runs: {len(rows)}\n')
print(f"{'model':<6} {'th':<5} {'gen':<12} {'eval':<5} {'baseline':>10} {'stego':>10} {'delta':>10}")
print('-' * 70)
for r in rows:
    if r.get('eval'):
        print(
            f"{r['model']:<6} {r['th']:<5} {r['status']:<12} {'yes':<5} "
            f"{r['baseline_ppl']:10.2f} {r['stego_ppl']:10.2f} {r['delta']:+10.2f}"
        )
    else:
        print(f"{r['model']:<6} {r['th']:<5} {r['status']:<12} {'no':<5}")